# Transformer

In [15]:
pip install torchdata torchtext portalocker

In [3]:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 43.4 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 146.8 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import torch
import torchtext
from torchtext.datasets import Multi30k
import inspect

print("PyTorch:", torch.__version__)
print("torchtext:", torchtext.__version__)
print("Multi30k:", Multi30k)
print("Multi30k module:", Multi30k.__module__)
print("Multi30k signature:", inspect.signature(Multi30k))

PyTorch: 2.14.1+cu130
torchtext: 0.6.0
Multi30k: <class 'torchtext.datasets.translation.Multi30k'>
Multi30k module: torchtext.datasets.translation
Multi30k signature: (path, exts, fields, **kwargs)


In [6]:
!pip install datasets

In [7]:
# 7.2 데이터세트 다운로드 및 전처리
from datasets import load_dataset
from collections import Counter

SRC_LANGUAGE = 'de'
TGT_LANGUAGE = 'en'
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

from torchtext.data.utils import get_tokenizer

token_transform = {
    SRC_LANGUAGE: get_tokenizer('spacy', language='de_core_news_sm'),
    TGT_LANGUAGE: get_tokenizer('spacy', language='en_core_web_sm'),
}

print('Token Transform:')
print(token_transform)


# Multi30k 데이터셋 다운로드
dataset = load_dataset("bentrevett/multi30k")

def Multi30k(split='train', language_pair=('de', 'en')):
    if split == 'valid':
        split = 'validation'

    for example in dataset[split]:
        yield (
            example['de'],
            example['en']
        )


def generate_tokens(text_iter, language):
    language_index = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1}

    for text in text_iter:
        yield token_transform[language](text[language_index[language]])


# 기존 build_vocab_from_iterator 기능 대체
class Vocab:
    def __init__(self, token_iterator, specials):
        counter = Counter()

        for tokens in token_iterator:
            counter.update(tokens)

        self.itos = list(specials) + [
            token for token, freq in counter.items()
            if token not in specials
        ]
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}
        self.default_index = UNK_IDX

    def set_default_index(self, index):
        self.default_index = index

    def __call__(self, tokens):
        return [self.stoi.get(token, self.default_index) for token in tokens]

    def __len__(self):
        return len(self.itos)

    def lookup_tokens(self, indices):
        return [self.itos[int(idx)] for idx in indices]


vocab_transform = {}

for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    train_iter = Multi30k(split='train', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))

    vocab_transform[language] = Vocab(
        generate_tokens(train_iter, language),
        specials=special_symbols
    )

for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    vocab_transform[language].set_default_index(UNK_IDX)

print('Vocab Transform:')
print(vocab_transform)

Token Transform:
{'de': functools.partial(<function _spacy_tokenize at 0x7dbf44b7f7e0>, spacy=<spacy.lang.de.German object at 0x7dbed37cd1d0>), 'en': functools.partial(<function _spacy_tokenize at 0x7dbf44b7f7e0>, spacy=<spacy.lang.en.English object at 0x7dbed37cf390>)}


README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Vocab Transform:
{'de': <__main__.Vocab object at 0x7dc01b7c2900>, 'en': <__main__.Vocab object at 0x7dbf30252ad0>}


In [8]:
#7.3 트랜스포머 모델 구성
import math
import torch
from torch import nn

class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(p=dropout)

    position = torch.arange(max_len).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )

    pe = torch.zeros(max_len, 1, d_model)
    pe[:, 0, 0::2] = torch.sin(position * div_term)
    pe[:, 0, 1::2] = torch.cos(position * div_term)
    self.register_buffer('pe', pe)

  def forward(self, x):
    x = x + self.pe[: x.size(0)]
    return self.dropout(x)

class TokenEmbedding(nn.Module):
  def __init__(self, vocab_size, emb_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, emb_size)
    self.emb_size = emb_size

  def forward(self, tokens):
    return self.embedding(tokens.long()) * math.sqrt(self.emb_size)

class Seq2SeqTransformer(nn.Module):
  def __init__(
      self,
      num_encoder_layers,
      num_decoder_layers,
      emb_size,
      max_len,
      nhead,
      src_vocab_size,
      tgt_vocab_size,
      dim_feedforward,
      dropout=0.1,
  ):
      super().__init__()
      self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
      self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
      self.positional_encoding = PositionalEncoding(
          d_model=emb_size, max_len=max_len, dropout=dropout
      )
      self.transformer = nn.Transformer(
          d_model=emb_size,
          nhead=nhead,
          num_encoder_layers=num_encoder_layers,
          num_decoder_layers=num_decoder_layers,
          dim_feedforward=dim_feedforward,
          dropout=dropout,
      )
      self.generator = nn.Linear(emb_size, tgt_vocab_size)

  def forward(
      self,
      src,
      trg,
      src_mask,
      tgt_mask,
      src_padding_mask,
      tgt_padding_mask,
      memory_key_padding_mask,
  ):
      src_emb = self.positional_encoding(self.src_tok_emb(src))
      tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))
      outs = self.transformer(
          src=src_emb,
          tgt=tgt_emb,
          src_mask=src_mask,
          tgt_mask=tgt_mask,
          memory_mask=None,
          src_key_padding_mask=src_padding_mask,
          tgt_key_padding_mask=tgt_padding_mask,
          memory_key_padding_mask=memory_key_padding_mask
      )
      return self.generator(outs)

  def encode(self, src, src_mask):
    return self.transformer.encoder(
        self.positional_encoding(self.src_tok_emb(src)), src_mask
    )

  def decode(self, tgt, memory, tgt_mask):
    return self.transformer.decoder(
        self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask
    )

In [9]:
#7.4 트랜스포머 모델 구조
from torch import optim

BATCH_SIZE = 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = Seq2SeqTransformer(
    num_encoder_layers=3,
    num_decoder_layers=3,
    emb_size=512,
    max_len=512,
    nhead=8,
    src_vocab_size=len(vocab_transform[SRC_LANGUAGE]),
    tgt_vocab_size=len(vocab_transform[TGT_LANGUAGE]),
    dim_feedforward=512,
).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX).to(DEVICE)
optimizer = optim.Adam(model.parameters())

for main_name, main_module in model.named_children():
  print(main_name)
  for sub_name, sub_module in main_module.named_children():
    print('L', sub_name)
    for ssub_name, ssub_module in sub_module.named_children():
      print('|  L', ssub_name)
      for sssub_name, sssub_module in ssub_module.named_children():
        print('|  | L', sssub_name)

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:145: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


src_tok_emb
L embedding
tgt_tok_emb
L embedding
positional_encoding
L dropout
transformer
L encoder
|  L layers
|  | L 0
|  | L 1
|  | L 2
|  L norm
L decoder
|  L layers
|  | L 0
|  | L 1
|  | L 2
|  L norm
generator


In [10]:
#7.5 배치 데이터 생성
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def sequential_transforms(*transforms):
  def func(txt_input):
    for transform in transforms:
      txt_input = transform(txt_input)
    return txt_input
  return func

def input_transform(token_ids):
  return torch.cat(
      (torch.tensor([BOS_IDX]), torch.tensor(token_ids), torch.tensor([EOS_IDX]))
  )

def collator(batch):
  src_batch, tgt_batch = [], []
  for src_sample, tgt_sample in batch:
    src_batch.append(text_transform[SRC_LANGUAGE](src_sample.rstrip('\n')))
    tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample.rstrip('\n')))

  src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
  tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
  return src_batch, tgt_batch

text_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
  text_transform[language] = sequential_transforms(
      token_transform[language], vocab_transform[language], input_transform
  )

data_iter = list(Multi30k(split='valid', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE)))
dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)
source_tensor, target_tensor = next(iter(dataloader))

print('(source, target):')
print(next(iter(data_iter)))

print('source_batch:', source_tensor.shape)
print(source_tensor)

print('target_batch:', target_tensor.shape)
print(target_tensor)

(source, target):
('Eine Gruppe von Männern lädt Baumwolle auf einen Lastwagen', 'A group of men are loading cotton onto a truck')
source_batch: torch.Size([35, 128])
tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [  70,   23,   23,  ...,   23,    4,   23],
        [ 130,   30,  250,  ...,   30, 4360,   24],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]])
target_batch: torch.Size([30, 128])
tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [  25,   25,   25,  ..., 1244,    4,   25],
        [ 255,   32,  246,  ...,   32, 3035,   26],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]])


In [11]:
#7.6 어텐션 마스크 생성
def generate_square_subsequent_mask(s):
    mask = torch.triu(
        torch.ones((s, s), device=DEVICE, dtype=torch.bool),
        diagonal=1
    )
    return mask


def create_mask(src, tgt):
    src_seq_len = src.shape[0]
    tgt_seq_len = tgt.shape[0]

    tgt_mask = generate_square_subsequent_mask(tgt_seq_len)
    src_mask = torch.zeros(
        (src_seq_len, src_seq_len), device=DEVICE, dtype=torch.bool
    )

    src_padding_mask = (src == PAD_IDX).transpose(0, 1).to(DEVICE)
    tgt_padding_mask = (tgt == PAD_IDX).transpose(0, 1).to(DEVICE)

    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask


source_tensor = source_tensor.to(DEVICE)
target_tensor = target_tensor.to(DEVICE)

target_input = target_tensor[:-1, :]
target_out = target_tensor[1:, :]


source_mask, target_mask, source_padding_mask, target_padding_mask = create_mask(
    source_tensor, target_input
)

print("source_mask:", source_mask.shape)
print(source_mask)
print("target_mask:", target_mask.shape)
print(target_mask)
print("source_padding_mask:", source_padding_mask.shape)
print(source_padding_mask)
print("target_padding_mask:", target_padding_mask.shape)
print(target_padding_mask)

source_mask: torch.Size([35, 35])
tensor([[False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        ...,
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False]], device='cuda:0')
target_mask: torch.Size([29, 29])
tensor([[False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True,  True,  True,
          Tr

In [12]:
#7.7 모델 학습 및 평가
def run(model, optimizer, criterion, split):
    model.train() if split == "train" else model.eval()

    data_iter = list(Multi30k(split=split, language_pair=(SRC_LANGUAGE, TGT_LANGUAGE)))
    dataloader = DataLoader(
        data_iter,
        batch_size=BATCH_SIZE,
        collate_fn=collator,
        shuffle=(split == "train")
    )

    losses = 0

    with torch.set_grad_enabled(split == "train"):
        for source_batch, target_batch in dataloader:
            source_batch = source_batch.to(DEVICE)
            target_batch = target_batch.to(DEVICE)

            target_input = target_batch[:-1, :]
            target_output = target_batch[1:, :]

            src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(
                source_batch, target_input
            )

            logits = model(
                src=source_batch,
                trg=target_input,
                src_mask=src_mask,
                tgt_mask=tgt_mask,
                src_padding_mask=src_padding_mask,
                tgt_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask,
            )

            loss = criterion(
                logits.reshape(-1, logits.shape[-1]),
                target_output.reshape(-1)
            )

            if split == "train":
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            losses += loss.item()

    return losses / len(dataloader)


for epoch in range(5):
    train_loss = run(model, optimizer, criterion, "train")
    val_loss = run(model, optimizer, criterion, "valid")
    print(f"Epoch: {epoch+1}, Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}")

Epoch: 1, Train loss: 4.682, Val loss: 3.901
Epoch: 2, Train loss: 3.761, Val loss: 3.614
Epoch: 3, Train loss: 3.494, Val loss: 3.510
Epoch: 4, Train loss: 3.336, Val loss: 3.448
Epoch: 5, Train loss: 3.227, Val loss: 3.432


In [13]:
#7.8 트랜스포머 모델 번역 결과
def greedy_decode(model, source_tensor, source_mask, max_len, start_symbol):
    source_tensor = source_tensor.to(DEVICE)
    source_mask = source_mask.to(DEVICE)

    memory = model.encode(source_tensor, source_mask)
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)
    for i in range(max_len - 1):
        memory = memory.to(DEVICE)
        target_mask = generate_square_subsequent_mask(ys.size(0))
        target_mask = target_mask.type(torch.bool).to(DEVICE)

        out = model.decode(ys, memory, target_mask)
        out = out.transpose(0, 1)
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys = torch.cat(
            [ys, torch.ones(1, 1).type_as(source_tensor.data).fill_(next_word)], dim=0
        )
        if next_word == EOS_IDX:
            break
    return ys


def translate(model, source_sentence):
    model.eval()
    source_tensor = text_transform[SRC_LANGUAGE](source_sentence).view(-1, 1)
    num_tokens = source_tensor.shape[0]
    src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool)
    tgt_tokens = greedy_decode(
        model, source_tensor, src_mask, max_len=num_tokens + 5, start_symbol=BOS_IDX
    ).flatten()
    output = vocab_transform[TGT_LANGUAGE].lookup_tokens(list(tgt_tokens.cpu().numpy()))[1:-1]
    return " ".join(output)


output_oov = translate(model, "Eine Gruppe von Menschen steht vor einem Iglu .")
output = translate(model, "Eine Gruppe von Menschen steht vor einem Gebäude .")
print(output_oov)
print(output)

A group of people are sitting on a bench .
A group of people are sitting on a bench .


# GPT

In [18]:
!pip install -q datasets

In [20]:
!wget -q https://nyu-mll.github.io/CoLA/cola_public_1.1.zip
!unzip -oq cola_public_1.1.zip

In [21]:
#7.11 CoLA 데이터세트 불러오기
import torch
import csv
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

def collator(batch, tokenizer, device):
    source, labels, texts = zip(*batch)
    tokenized = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        return_tensors="pt"
    )
    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)
    labels = torch.tensor(labels, dtype=torch.long).to(device)
    return input_ids, attention_mask, labels


# CoLA 원본 데이터셋 읽기
def load_cola(path):
    data = []

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")

        for row in reader:
            source = row[0]
            label = int(row[1])
            text = row[3]

            data.append((source, label, text))

    return data


# 학습 및 검증 데이터
train_data = load_cola("cola_public/raw/in_domain_train.tsv")
valid_data = load_cola("cola_public/raw/in_domain_dev.tsv")

# 공식 테스트 데이터에는 정답 레이블이 없으므로
# 학습 데이터에서 별도의 테스트 데이터를 분리
train_data, test_data = train_test_split(
    train_data,
    test_size=0.1,
    random_state=42,
    stratify=[item[1] for item in train_data]
)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

epochs = 3
batch_size = 16
device = "cuda" if torch.cuda.is_available() else "cpu"

train_dataloader = DataLoader(
    train_data,
    batch_size=batch_size,
    collate_fn=lambda x: collator(x, tokenizer, device),
    shuffle=True,
)
valid_dataloader = DataLoader(
    valid_data,
    batch_size=batch_size,
    collate_fn=lambda x: collator(x, tokenizer, device)
)
test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    collate_fn=lambda x: collator(x, tokenizer, device)
)

print("Train Dataset Length :", len(train_data))
print("Valid Dataset Length :", len(valid_data))
print("Test Dataset Length :", len(test_data))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Train Dataset Length : 7695
Valid Dataset Length : 527
Test Dataset Length : 856


In [22]:
#7.12 GPT-2 모델 설정
from torch import optim
from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path='gpt2',
    num_labels=2
).to(device)
model.config.pad_token_id = model.config.eos_token_id
optimizer = optim.Adam(model.parameters(), lr=5e-5)

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
#7.13 GPT-2 모델 학습 및 검증
import numpy as np
import os
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion = nn.CrossEntropyLoss()
        val_loss, val_accuracy = 0.0, 0.0

        for input_ids, attention_mask, labels in dataloader:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            logits = outputs.logits

            loss = criterion(logits, labels)
            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()
            accuracy = calc_accuracy(logits, label_ids)

            val_loss += loss
            val_accuracy += accuracy

    val_loss = val_loss/len(dataloader)
    val_accuracy = val_accuracy/len(dataloader)
    return val_loss, val_accuracy


# 모델 저장 폴더 생성
os.makedirs("../models", exist_ok=True)

best_loss = 10000
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_accuracy = evaluation(model, valid_dataloader)
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Accuracy {val_accuracy:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "../models/GPT2ForSequenceClassification.pt")
        print("Saved the model weights")

Epoch 1: Train Loss: 0.6069 Val Loss: 0.6195 Val Accuracy 0.6924
Saved the model weights
Epoch 2: Train Loss: 0.6014 Val Loss: 0.6062 Val Accuracy 0.6944
Saved the model weights
Epoch 3: Train Loss: 0.5790 Val Loss: 0.5908 Val Accuracy 0.6997
Saved the model weights


In [25]:
#7.14 모델 평가
model = GPT2ForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="gpt2",
    num_labels=2
).to(device)
model.config.pad_token_id = model.config.eos_token_id
model.load_state_dict(torch.load("../models/GPT2ForSequenceClassification.pt"))

test_loss, test_accuracy = evaluation(model, test_dataloader)
print(f"Test Loss : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Test Loss : 0.6205
Test Accuracy : 0.7072
